In [3]:
# REVIEWER 1 — STAGE 1: ROOT-BOUND DISTRIBUTIONS (ONE CELL, CSV ONLY; NO OPTIMIZATION)
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json

ROOT = Path(r'C:\Users\suuser\Desktop\minlplib')
PAPER = ROOT / 'paperexperiments'
PRIME = PAPER / 'synthetic_prime_minimal_revision'
COMBINED = ROOT / 'Areview' / 'combined_minlplib_qplib'
OUT = PAPER / 'reviewer_synthetic_difficulty' / 'stage1_root_distributions'
OUT.mkdir(parents=True, exist_ok=True)
TARGET_PRIME = PRIME/'bb_separate_trees'/'synthetic_prime_8016_fixed100_v1'/'postrun_clean'/'targets'/'method2'/'synthetic_prime'/'targets_method2_relaxation_synthetic_prime_probe10.csv'
TARGET_COMBINED = COMBINED/'targets'/'method2'/'combined_minlplib_qplib'/'targets_method2_relaxation_combined_minlplib_qplib_probe10.csv'
PRIME_ROOT = PRIME/'instances_mixed_8016_root_results.csv'
# NOTE: primary root near-tie threshold is explicitly a descriptive normalized bound-difference threshold.
# It is NOT asserted to equal the manuscript's ML delta, whose root-target formula must be independently verified.
TIE_LEVELS = [0.001, 0.01, 0.05]
PRIMARY_TIE = 0.01
EPS = 1e-3
EXCLUDE_ROOT_ONLY = {5049}
EXCLUDE_COMBINED = {'QPLIB||QPLIB_2512','QPLIB||QPLIB_3402','QPLIB||QPLIB_3703',
                    'MINLPLIB||ex5_3_3','QPLIB||QPLIB_9048'}

def csv(path):
    if not path.is_file(): raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path, low_memory=False)

def numid(series):
    x=series.astype('string').str.strip().str.replace(r'^SYNTHETIC\|\|','',regex=True)
    if x.isna().any() or not x.str.fullmatch(r'\d+').all():
        raise ValueError('Invalid Synthetic prime IDs; no implicit positional matching')
    return x.astype(int)

def tidy_bounds(df, label, family=''):
    cols=['lp_obj','sdp_obj','sdp_prime_obj','lp_t','sdp_t','sdp_prime_t']
    missing=[c for c in cols[:3] if c not in df.columns]
    if missing: raise ValueError(f'{label}: missing bound columns {missing}')
    z=df.copy()
    for c in cols:
        if c not in z: z[c]=np.nan
        z[c]=pd.to_numeric(z[c],errors='coerce')
    for c in ('lp','sdp','sdp_prime'):
        status=f'{c}_status'
        z[f'{c}_finite']=np.isfinite(z[f'{c}_obj']) & ((z[status].astype(str).str.lower()=='optimal') if status in z else True)
    z['dataset']=label
    z['family']=family
    return z

def dataset_analysis(df, label):
    summaries=[]; details=[]
    for other,short in [('sdp','SDP'),('sdp_prime','SDPprime')]:
        valid=df['lp_finite'] & df[other+'_finite']
        d=df.loc[valid].copy()
        lp=d['lp_obj'].to_numpy(dtype=float); alt=d[other+'_obj'].to_numpy(dtype=float)
        # Symmetric, dimensionless *descriptive* separation (not the target delta).
        sep=(alt-lp)/(np.maximum.reduce([np.abs(lp),np.abs(alt),np.full(len(d),EPS)]))
        d['normalized_bound_difference']=sep
        d['absolute_normalized_difference']=np.abs(sep)
        d['comparison']='LP_vs_'+short
        d['stronger']=np.where(sep>PRIMARY_TIE,short,np.where(sep < -PRIMARY_TIE,'LP','near_tie'))
        d['dataset']=label
        details.append(d)
        row={'dataset':label, 'comparison':'LP_vs_'+short, 'total_rows':len(df),
             'valid_finite_pairs':len(d),'invalid_or_missing_pairs':len(df)-len(d),
             'LP_wins_fraction':float(np.mean(sep < -PRIMARY_TIE)) if len(d) else np.nan,
             'alternative_wins_fraction':float(np.mean(sep > PRIMARY_TIE)) if len(d) else np.nan,
             'median_absolute_relative_difference':float(np.median(np.abs(sep))) if len(d) else np.nan,
             'q90_absolute_relative_difference':float(np.quantile(np.abs(sep),0.90)) if len(d) else np.nan,
             'median_signed_relative_difference':float(np.median(sep)) if len(d) else np.nan,
             'LP_median_time_s':float(df['lp_t'].median()),
             'alternative_median_time_s':float(df[other+'_t'].median()),
             'full_SDP_finite_count':int(df['sdp_finite'].sum()),
             'LP_finite_count':int(df['lp_finite'].sum()),
             'SDPprime_finite_count':int(df['sdp_prime_finite'].sum())}
        for tol in TIE_LEVELS: row[f'near_tie_fraction_at_{tol:g}']=float(np.mean(np.abs(sep)<=tol)) if len(d) else np.nan
        summaries.append(row)
    return summaries,details

# Synthetic prime: authoritative unbalanced B&B population (6069), minus root-only ID 5049 => 6068.
prime=tidy_bounds(csv(PRIME_ROOT),'Synthetic_prime')
if prime['id'].duplicated().any(): raise RuntimeError('Duplicate Synthetic prime root IDs')
target=csv(TARGET_PRIME)
idcol=next((c for c in ('instance_key','dataset_id','instance_id','id') if c in target),None)
if idcol is None: raise RuntimeError('No authoritative ID column in final target CSV')
ids=numid(target[idcol])
if len(ids)!=6069 or ids.duplicated().any():
    raise RuntimeError(f'Expected 6069 distinct unbalanced final B&B Synthetic prime IDs, got {len(ids)} / unique {ids.nunique()}')
prime['id']=pd.to_numeric(prime['id'],errors='raise').astype(int)
root_ids=set(ids)-EXCLUDE_ROOT_ONLY
if len(root_ids)!=6068: raise RuntimeError('Root-only exclusion did not produce 6068 IDs')
prime=prime[prime['id'].isin(root_ids)].copy()
if len(prime)!=6068: raise RuntimeError(f'Only {len(prime)} Synthetic prime root rows matched 6068 IDs')
prime['key']='SYNTHETIC||'+prime['id'].astype(str)
print('Synthetic prime: 6068 unbalanced final ROOT instances (not 4832 balanced; not 8016 candidates)')

# Original synthetic: all individual-family objective CSVs. Exclude mixed_train8 (overlapping/derived population).
orig_paths=sorted(p for p in PAPER.glob('instancesobj_*.csv') if p.name!='instancesobj_mixed_train8.csv' and not p.stem.removeprefix('instancesobj_').endswith('a') and p.stem.removeprefix('instancesobj_') not in {'5_100_1200', '5_100_1200a'})
if not orig_paths: raise FileNotFoundError('No individual-family instancesobj_*.csv files found')
original=[]
for p in orig_paths:
    family=p.stem.removeprefix('instancesobj_')
    d=tidy_bounds(csv(p),'Synthetic_original',family)
    if 'id' not in d: raise ValueError(f'{p.name}: missing id')
    if d['id'].duplicated().any(): raise RuntimeError(f'Duplicate local IDs in {p.name}')
    d['key']=family+'||'+d['id'].astype(str)
    original.append(d)
orig=pd.concat(original,ignore_index=True)
if orig['key'].duplicated().any(): raise RuntimeError('Duplicated family+ID original synthetic keys')
print('Original Synthetic individual-family CSVs (excluding *a families):',len(orig_paths),'rows:',len(orig))
print('Excluded mixed_train8, every family name ending in a, and 5_100_1200/5_100_1200a explicitly; this is a GENERATOR-FAMILY population, not asserted to match training exactly')

# Combined: target records define final population, but may not themselves contain root objective columns.
combined_target=csv(TARGET_COMBINED)
if 'instance_key' in combined_target:
    combined_target['key']=combined_target['instance_key'].astype(str).str.strip()
elif {'source','instance_name'}.issubset(combined_target.columns):
    combined_target['key']=combined_target['source'].astype(str).str.upper()+'||'+combined_target['instance_name'].astype(str).str.replace(r'\.pkl$','',regex=True)
else: raise ValueError('Combined target has no collision-safe key')
combined_target=combined_target[~combined_target['key'].isin(EXCLUDE_COMBINED)].copy()
if combined_target['key'].duplicated().any(): raise RuntimeError('Duplicate Combined keys')
print('Combined target eligible keys (after known exclusions):',len(combined_target))
# We do NOT assume B&B delta is a root delta, or that arbitrary summary rows are root bound columns.
# Seek an existing root-objective table only if its schema and exact identity keys can be checked.
combined_source=None
combined_root=None
candidates=[COMBINED/'summary_combined_minlplib_qplib_BB_clean.csv',COMBINED/'combined_minlplib_qplib_root_results.csv']
for p in candidates:
    if not p.exists(): continue
    c=csv(p)
    if not {'lp_obj','sdp_obj','sdp_prime_obj'}.issubset(c.columns): continue
    if 'instance_key' in c: c['key']=c['instance_key'].astype(str).str.strip()
    elif {'source','instance_name'}.issubset(c.columns):
        c['key']=c['source'].astype(str).str.upper()+'||'+c['instance_name'].astype(str).str.replace(r'\.pkl$','',regex=True)
    else: continue
    if c['key'].duplicated().any(): continue
    matched=c[c['key'].isin(set(combined_target['key']))].copy()
    if len(matched)!=len(combined_target): continue
    combined_root=tidy_bounds(matched,'Combined'); combined_source=p
    break
if combined_root is None:
    print('WARNING: No schema-verified Combined root bounds CSV found. Combined root-gap distribution is NOT fabricated.')
    print('Combined ML B&B delta is available in the target file but is a DISTINCT quantity.')
else:
    print('Verified Combined root bound source:',combined_source)

summary=[]; item_rows=[]
for df,name in [(orig,'Synthetic_original_all_families'),(prime,'Synthetic_prime_final_root')]:
    a,b=dataset_analysis(df,name); summary+=a; item_rows+=b
for family,g in orig.groupby('family',sort=True):
    a,_=dataset_analysis(g,'Synthetic_original_family_'+family); summary+=a
if combined_root is not None:
    a,b=dataset_analysis(combined_root,'Combined_final_root'); summary+=a; item_rows+=b
summary_df=pd.DataFrame(summary)
summary_df.to_csv(OUT/'root_bound_summary.csv',index=False)
items=pd.concat(item_rows,ignore_index=True)
cols=[c for c in ['dataset','family','key','id','id_global','comparison','lp_obj','sdp_obj','sdp_prime_obj',
                 'normalized_bound_difference','absolute_normalized_difference','stronger'] if c in items]
items[cols].to_csv(OUT/'root_bound_instance_level.csv',index=False)
# Save EXACT supplied B&B delta distributions separately, with explicit labeling.
bb_delta=[]
for df,label in [(target,'Synthetic_prime_BB_6069_unbalanced'),(combined_target,'Combined_BB_final')]:
    if 'delta' in df:
        arr=pd.to_numeric(df['delta'],errors='coerce').to_numpy(dtype=float)
        arr=arr[np.isfinite(arr)]
        bb_delta.append({'dataset':label,'N':len(arr),'median_delta':float(np.median(arr)),
              'median_abs_delta':float(np.median(np.abs(arr))),
              'frac_abs_delta_le_0.001':float(np.mean(np.abs(arr)<=0.001)),
              'frac_abs_delta_le_0.01':float(np.mean(np.abs(arr)<=0.01)),
              'q10_delta':float(np.quantile(arr,.10)),'q90_delta':float(np.quantile(arr,.90))})
if bb_delta: pd.DataFrame(bb_delta).to_csv(OUT/'BB_target_delta_SEPARATE_from_root.csv',index=False)
# Figures: outliers can be extreme, so use clipped x limits and report clipping explicitly.
for comparison in ['LP_vs_SDP','LP_vs_SDPprime']:
    fig,ax=plt.subplots(figsize=(9,5))
    for label in ['Synthetic_original_all_families','Synthetic_prime_final_root','Combined_final_root']:
        d=items[(items.dataset==label)&(items.comparison==comparison)]
        if d.empty: continue
        x=d['normalized_bound_difference'].dropna().to_numpy(dtype=float)
        ax.hist(x,bins=np.linspace(-1,1,81),density=True,alpha=.35,label=f'{label} (N={len(x)})')
    ax.axvline(-PRIMARY_TIE,ls='--',lw=1,color='black'); ax.axvline(PRIMARY_TIE,ls='--',lw=1,color='black')
    ax.set(xlabel='Signed normalized alternative bound minus LP bound',ylabel='Density',
           title=comparison+' | descriptive root bound differences (NOT paper target delta)',xlim=(-1,1))
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT/f'{comparison}_root_difference_hist.png',dpi=180); plt.close(fig)
metadata={'primary_tie_threshold':PRIMARY_TIE,'sensitivity_thresholds':TIE_LEVELS,
          'root_difference_formula':'(L_alternative-L_LP)/max(|L_alternative|,|L_LP|,0.001)',
          'paper_delta_formula_verified':False,'BB_delta_is_not_root_delta':True,
          'original_family_files':[p.name for p in orig_paths],
          'original_mixed_train8_excluded':True,
          'synthetic_prime_population':'6068 root-only = 6069 final unbalanced B&B IDs minus id 5049',
          'combined_root_bound_source':str(combined_source) if combined_source else None,
          'no_global_nonconvex_solves':True,
          'limitations':['Original generator-family aggregate may differ from manuscript training population',
                         'Root comparison is descriptive normalized bound separation, not verified paper delta',
                         'Full SDP -inf and missing values excluded only from finite paired comparisons and counted in coverage']}
(OUT/'analysis_audit.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('\n=== SUMMARY ===')
print(summary_df.loc[summary_df.dataset.isin(['Synthetic_original_all_families','Synthetic_prime_final_root','Combined_final_root'])].to_string(index=False))
print('\nSAVED IN:',OUT)
for p in sorted(OUT.iterdir()): print(' ',p.name)
print('Send me root_bound_summary.csv, analysis_audit.json, and BB_target_delta_SEPARATE_from_root.csv (if generated).')


Synthetic prime: 6068 unbalanced final ROOT instances (not 4832 balanced; not 8016 candidates)
Original Synthetic individual-family CSVs (excluding *a families): 11 rows: 320500
Excluded mixed_train8, every family name ending in a, and 5_100_1200/5_100_1200a explicitly; this is a GENERATOR-FAMILY population, not asserted to match training exactly
Combined target eligible keys (after known exclusions): 278
Combined ML B&B delta is available in the target file but is a DISTINCT quantity.

=== SUMMARY ===
                        dataset     comparison  total_rows  valid_finite_pairs  invalid_or_missing_pairs  LP_wins_fraction  alternative_wins_fraction  median_absolute_relative_difference  q90_absolute_relative_difference  median_signed_relative_difference  LP_median_time_s  alternative_median_time_s  full_SDP_finite_count  LP_finite_count  SDPprime_finite_count  near_tie_fraction_at_0.001  near_tie_fraction_at_0.01  near_tie_fraction_at_0.05
Synthetic_original_all_families      LP_vs_SDP

In [6]:
# STAGE 2 — SYNTHETIC PRIME-ONLY CHECKPOINT RESUME (ONE CELL)
# IMPORTANT: restart Jupyter kernel first, then run this as the first cell.
# Adds 24 Original Synthetic + 24 Synthetic-prime cases (300s each) and retries the 3 prior extraction errors.
# Does NOT rerun successful older cases; keeps original 600s results separately and preserves all files.
from pathlib import Path
import sys, pickle, gc, time, json
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

ROOT = Path(r'C:\Users\suuser\Desktop\minlplib\paperexperiments')
PRIME = ROOT / 'synthetic_prime_minimal_revision'
BASE = ROOT / 'reviewer_synthetic_difficulty'
OUT = BASE / 'stage2_global_extension'
OUT.mkdir(parents=True, exist_ok=True)
OLD_RESULTS = BASE / 'global_qcqp_results.csv'
OLD_MANIFEST = BASE / 'sample_manifest.csv'
FINAL_IDS_FILE = PRIME / 'bb_separate_trees' / 'synthetic_prime_8016_fixed100_v1' / 'postrun_clean' / 'targets' / 'method2' / 'synthetic_prime' / 'targets_method2_relaxation_synthetic_prime_probe10.csv'
PRIME_CSV = PRIME / 'instances_mixed_8016_root_results.csv'
PRIME_PKL = PRIME / 'instances_mixed_8016_candidate.pkl'
SEED = 20261025
NEW_PER_DATASET = 24   # total new: 48. Combined with original 48 -> 96 (plus error retries)
TIME_LIMIT_NEW = 300   # seconds; old solves used 600s, so summarize protocols separately
TIME_LIMIT_RETRY = 300
THREADS = 1
# All available NON-a, NON-1200, NON-mixed original families. Include families absent from initial 6-family sample.
FAMILIES = ['10_1_50','10_2_10','10_2_40','10_2_50','20_2_10','5_100_500','5_10_10','5_1_50','5_2_10','5_2_40','5_2_50']

class QCQPInstance: pass
sys.modules['__main__'].QCQPInstance = QCQPInstance
rng = np.random.default_rng(SEED)
for p in (OLD_RESULTS,OLD_MANIFEST,FINAL_IDS_FILE,PRIME_CSV,PRIME_PKL):
    if not p.is_file(): raise FileNotFoundError(f'Required existing file missing: {p}')
old = pd.read_csv(OLD_RESULTS,low_memory=False)
oldm = pd.read_csv(OLD_MANIFEST,low_memory=False)
if len(old)!=48 or len(oldm)!=48: raise RuntimeError(f'Expected 48 original results/manifest rows, got {len(old)}/{len(oldm)}. Stop to avoid resampling.')
if old[['dataset','pickle_file','instance_id']].duplicated().any(): raise RuntimeError('Prior result identities are duplicated')
if oldm[['dataset','pickle_file','id']].duplicated().any(): raise RuntimeError('Prior manifest identities duplicated')
oldkeys=set(zip(old.dataset.astype(str),old.pickle_file.astype(str),pd.to_numeric(old.instance_id).astype(int)))
old_manifest_keys=set(zip(oldm.dataset.astype(str),oldm.pickle_file.astype(str),pd.to_numeric(oldm.id).astype(int)))
if oldkeys != old_manifest_keys: raise RuntimeError('Prior results and prior sample manifest identities disagree')
old_status=old.status.astype(str).str.lower()
retry_old=old.loc[old_status.eq('error')].copy()
print('Prior benchmark:',len(old),'rows | errors to retry:',len(retry_old),'| original 600s results protected')

def candidate_frame(path, ds, n_needed, excluded_ids):
    x=pd.read_csv(path,low_memory=False)
    if x.id.duplicated().any(): raise ValueError(f'Duplicate ids: {path.name}')
    x['id']=pd.to_numeric(x['id'],errors='raise').astype(int)
    for col in ('lp_obj','sdp_prime_obj','sdp_obj'):
        x[col]=pd.to_numeric(x[col],errors='coerce')
    bounded=x.bounds_exist.astype(str).str.lower().eq('true')
    valid=(x.lp_status.astype(str).str.lower().eq('optimal') &
           x.sdp_prime_status.astype(str).str.lower().eq('optimal') &
           np.isfinite(x.lp_obj) & np.isfinite(x.sdp_prime_obj) & bounded)
    x=x.loc[valid & ~x.id.isin(excluded_ids)].copy()
    if len(x)<n_needed: raise RuntimeError(f'{ds}: {len(x)} eligible new rows but need {n_needed}')
    x['signed_difference']=x.sdp_prime_obj-x.lp_obj
    x['rel_difference']=x.signed_difference/(np.maximum.reduce([np.abs(x.lp_obj.to_numpy()),np.abs(x.sdp_prime_obj.to_numpy()),np.full(len(x),1e-3)]))
    x['magnitude_bin']=pd.cut(np.abs(x.rel_difference),[-0.000001,.01,.05,.20,np.inf],labels=['tie_1pct','small','medium','large'])
    x['winner_bin']=np.where(x.rel_difference > .01,'SDPprime',np.where(x.rel_difference < -.01,'LP','near_tie'))
    return x

# Use the authoritative 6069 B&B IDs, minus 5049: exactly the 6068 final unbalanced ROOT IDs.
prime_csv=pd.read_csv(PRIME_CSV,low_memory=False)
prime_csv['id']=pd.to_numeric(prime_csv.id,errors='raise').astype(int)
if prime_csv.id.duplicated().any(): raise RuntimeError('Synthetic-prime root CSV id duplication')
targ=pd.read_csv(FINAL_IDS_FILE,low_memory=False)
if 'instance_key' not in targ.columns: raise RuntimeError('Final target file missing instance_key; do not infer row positions')
raw=targ.instance_key.astype(str).str.strip().str.replace(r'^SYNTHETIC\|\|','',regex=True)
if not raw.str.fullmatch(r'\d+').all(): raise RuntimeError('Final target keys are not numeric Synthetic-prime IDs')
ids=pd.to_numeric(raw).astype(int)
if len(ids)!=6069 or ids.duplicated().any(): raise RuntimeError(f'Expected 6069 unique Synthetic-prime B&B IDs, found {len(ids)}')
final_ids=set(ids)-{5049}
if len(final_ids)!=6068 or len(set(prime_csv.id)&final_ids)!=6068: raise RuntimeError('Final Synthetic-prime root ID reconciliation failed')
prior_prime_ids={id_ for ds,filename,id_ in oldkeys if ds=='Synthetic_prime'}
new_prime_frame=candidate_frame(PRIME_CSV,'Synthetic_prime',NEW_PER_DATASET,prior_prime_ids)
new_prime_frame=new_prime_frame[new_prime_frame.id.isin(final_ids)].copy()
if len(new_prime_frame)<NEW_PER_DATASET: raise RuntimeError('Not enough eligible final Synthetic-prime candidates')

# Stratify Synthetic-prime across variable dimension and root-difference size.
def round_robin_sample(df,n,group_cols,rng):
    groups=[]
    for _,g in df.groupby(group_cols,observed=True,sort=True):
        order=rng.permutation(len(g))
        groups.append(g.iloc[order].to_dict('records'))
    rng.shuffle(groups)
    result=[]
    while len(result)<n:
        progressed=False
        for group in groups:
            if group and len(result)<n:
                result.append(group.pop());progressed=True
        if not progressed:raise RuntimeError('Insufficient candidates in round-robin sampler')
    return result

jobs=[]
prime_select=round_robin_sample(new_prime_frame,NEW_PER_DATASET,['n','magnitude_bin'],rng)
for r in prime_select:jobs.append({'dataset':'Synthetic_prime','pickle_file':PRIME_PKL.name,'id':int(r['id']),'id_global':r['id_global'],'source':'new_300s','family':'synthetic_prime'})

# Spread 24 Original Synthetic additions over all 11 families, prioritizing the 5 not used in the previous sample.
prior_family={p.replace('instances_','').replace('.pkl','') for ds,p,i in oldkeys if ds=='Synthetic'}
alloc={f:2 for f in FAMILIES}  # 22 slots, guaranteed representation of every family in extension
for f in [f for f in FAMILIES if f not in prior_family][:2]: alloc[f]+=1  # 24 slots total
assert sum(alloc.values())==NEW_PER_DATASET
for family in FAMILIES:
    pkl=ROOT/f'instances_{family}.pkl'
    csv=ROOT/f'instancesobj_{family}.csv'
    if not pkl.exists() or not csv.exists():raise FileNotFoundError(f'Missing original family pickle/CSV: {pkl} / {csv}')
    excl={i for ds,p,i in oldkeys if ds=='Synthetic' and p==pkl.name}
    frame=candidate_frame(csv,family,alloc[family],excl)
    rows=round_robin_sample(frame,alloc[family],['magnitude_bin','winner_bin'],rng)
    for r in rows:jobs.append({'dataset':'Synthetic','pickle_file':pkl.name,'id':int(r['id']),'id_global':r['id_global'],'source':'new_300s','family':family})

# Correct ONLY the three old result-extraction errors, leaving old completed solves untouched.
for _,r in retry_old.iterrows():
    jobs.append({'dataset':str(r.dataset),'pickle_file':str(r.pickle_file),'id':int(r.instance_id),
                 'id_global':r.id_global,'source':'retry_old_error_300s','family':str(r.pickle_file)})
manifest=pd.DataFrame(jobs)
if manifest[['dataset','pickle_file','id']].duplicated().any():raise RuntimeError('Duplicate new jobs')
new=manifest[manifest.source.eq('new_300s')]
if len(new)!=48 or new.dataset.value_counts().to_dict()!={'Synthetic':24,'Synthetic_prime':24}:raise RuntimeError('New sample size mismatch')
for _,r in new.iterrows():
    if (r.dataset,r.pickle_file,int(r.id)) in oldkeys: raise RuntimeError('New sample duplicates old sample')
manifest_path=OUT/'stage2_manifest.csv'
if manifest_path.exists():
    before=pd.read_csv(manifest_path).fillna('').astype(str)
    after=manifest.fillna('').astype(str)
    if not before.equals(after):
        raise RuntimeError('Existing Stage 2 manifest differs from this selection; refuse resampling. Preserve old run and use the original stage2_manifest.csv.')
else:manifest.to_csv(manifest_path,index=False)
print('Stage 2 manifest:',manifest_path,'new:',len(new),'retries:',len(retry_old))

# Gurobi model is ORIGINAL nonconvex QCQP; saved LP/SDP/SDPprime values are NOT recomputed.
def solve_original(instance, limit_s):
    As=[np.asarray(a,dtype=float) for a in instance.A_list]
    bs=[np.asarray(b,dtype=float).reshape(-1) for b in instance.b_list]
    cs=[float(np.asarray(c).reshape(-1)[0]) for c in instance.c_list]
    lo=np.asarray(instance.l,dtype=float).reshape(-1); hi=np.asarray(instance.u,dtype=float).reshape(-1)
    n=len(lo)
    if not(len(As)==len(bs)==len(cs) and all(a.shape==(n,n) for a in As) and all(len(b)==n for b in bs)):
        raise ValueError('Unexpected QCQP dimensions')
    if not(np.isfinite(lo).all() and np.isfinite(hi).all() and (lo<=hi).all()):
        raise ValueError('Invalid/nonfinite ORIGINAL bounds; do not substitute artificial bounds')
    model=gp.Model('nonconvex_QCQP_global')
    try:
        model.Params.OutputFlag=0;model.Params.NonConvex=2;model.Params.TimeLimit=limit_s;model.Params.Threads=THREADS
        x=model.addVars(n,lb=lo.tolist(),ub=hi.tolist(),name='x')
        def expr(k):
            A=(As[k]+As[k].T)*.5
            q=gp.QuadExpr()
            for i in np.flatnonzero(np.diag(A)):
                q.add(float(A[i,i])*x[int(i)]*x[int(i)])
            ii,jj=np.nonzero(np.triu(A,1))
            for i,j in zip(ii,jj): q.add(float(2*A[i,j])*x[int(i)]*x[int(j)])
            linear=gp.quicksum(float(2*bs[k][i])*x[i] for i in range(n) if bs[k][i]!=0)
            return q+linear+cs[k]
        model.setObjective(expr(0),GRB.MINIMIZE)
        for k in range(1,len(As)):model.addQConstr(expr(k)<=0,name=f'qc_{k}')
        model.optimize()
        code=int(model.Status)
        out=dict(status_code=code,status={GRB.OPTIMAL:'optimal',GRB.TIME_LIMIT:'time_limit',GRB.INFEASIBLE:'infeasible',GRB.UNBOUNDED:'unbounded',GRB.INF_OR_UNBD:'inf_or_unbd',GRB.NUMERIC:'numeric'}.get(code,'other'),
                 runtime_s=float(model.Runtime),nodes=float(model.NodeCount),sol_count=int(model.SolCount),n=n,m_constr=len(As)-1,
                 best_feasible_obj=np.nan,best_global_bound=np.nan,gurobi_mip_gap=np.nan,error='')
        if model.SolCount>0:
            try:out['best_feasible_obj']=float(model.ObjVal)
            except (AttributeError,gp.GurobiError):pass
            try:out['gurobi_mip_gap']=float(model.MIPGap)
            except (AttributeError,gp.GurobiError):pass
        # Critical fix: ObjBound is not always available even when model.Status is not infeasible.
        try:out['best_global_bound']=float(model.ObjBound)
        except (AttributeError,gp.GurobiError):pass
        return out
    finally:model.dispose()

def instance_at(objects, row):
    idx=int(row.id)
    if idx<0 or idx>=len(objects):raise RuntimeError(f'ID {idx} not an index in pickle')
    inst=objects[idx]
    ident=str(inst.instance_id)
    if ident not in {str(idx),str(row.id_global)} and not str(row.id_global).startswith(ident+'_'):
        raise RuntimeError(f'Pickle ID does not match CSV for {idx}: {ident}, {row.id_global}')
    return inst

# Resume from checkpoint if interrupted. Do not overwrite successful records or solve old completed results again.
checkpoint=OUT/'stage2_new_and_retries_results.csv'
if checkpoint.exists():
    done=pd.read_csv(checkpoint,low_memory=False)
    if done[['dataset','pickle_file','instance_id','source']].duplicated().any():raise RuntimeError('Duplicate checkpoint rows')
else:done=pd.DataFrame()
finished=set(zip(done.dataset.astype(str),done.pickle_file.astype(str),pd.to_numeric(done.instance_id).astype(int),done.source.astype(str))) if len(done) else set()
# IMPORTANT: RUN IN A FRESH JUPYTER KERNEL. This pass loads ONLY Synthetic-prime.
# All 24 new Original Synthetic runs and both original retries are already checkpointed.
# The 3GB prime pickle was successfully unpickled in the initial difficulty experiment.
# Avoid loading any original family pickle in the same kernel beforehand.
ordered_files = [PRIME_PKL.name]
files_deferred = []
for filename in ordered_files:
    pending=manifest[(manifest.pickle_file==filename) & ~manifest.apply(lambda r:(r.dataset,r.pickle_file,int(r.id),r.source) in finished,axis=1)]
    if pending.empty:continue
    path=(PRIME if filename==PRIME_PKL.name else ROOT)/filename
    print('Loading',filename,'for',len(pending),'unfinished jobs',flush=True)
    try:
        # This may require much more RAM than the compressed/on-disk pickle size.
        with path.open('rb') as stream:
            objects=pickle.load(stream)
    except MemoryError:
        print('MEMORY LIMIT: cannot load',filename,
              '| all previously solved instances are checkpointed; file deferred.',flush=True)
        files_deferred.append(filename)
        gc.collect()
        continue
    root_frame=prime_csv if filename==PRIME_PKL.name else pd.read_csv(ROOT/f'instancesobj_{filename.removeprefix("instances_").removesuffix(".pkl")}.csv',low_memory=False)
    root_frame['id']=pd.to_numeric(root_frame.id,errors='raise').astype(int)
    lookup=root_frame.set_index('id')
    for row in pending.itertuples(index=False):
        if int(row.id) not in lookup.index:raise RuntimeError(f'Cannot match root values: {filename}, {row.id}')
        r=lookup.loc[int(row.id)]
        res=dict(dataset=row.dataset,pickle_file=filename,instance_id=int(row.id),id_global=row.id_global,
                 source=row.source,family=row.family,time_limit_s=TIME_LIMIT_NEW if row.source=='new_300s' else TIME_LIMIT_RETRY,
                 lp_bound=r.get('lp_obj'),sdp_bound=r.get('sdp_obj'),sdpp_bound=r.get('sdp_prime_obj'))
        try:res.update(solve_original(instance_at(objects,row),res['time_limit_s']))
        except Exception as e:res.update(status='error',error=f'{type(e).__name__}: {e}')
        done=pd.concat([done,pd.DataFrame([res])],ignore_index=True)
        done.to_csv(checkpoint,index=False)
        finished.add((row.dataset,filename,int(row.id),row.source))
        print(row.dataset,row.id_global,'->',res['status'],f"{res.get('runtime_s',float('nan')):.2f}s",flush=True)
    del objects;gc.collect()

# Keep original 48 rows AS IS. Retry results are additional records, not retroactively changed 600s outcomes.
combined=pd.concat([old.assign(source='original_600s',time_limit_s=600),done],ignore_index=True,sort=False)
combined.to_csv(OUT/'stage2_all_results_with_original_and_retries.csv',index=False)
summary=combined.groupby(['dataset','source','status'],dropna=False).size().rename('count').reset_index()
expected_jobs=len(manifest)
completed_jobs=len(done)
summary.attrs['completed_jobs']=completed_jobs
print(f'Stage 2 new/retry jobs completed: {completed_jobs}/{expected_jobs}.')
summary.to_csv(OUT/'stage2_status_summary.csv',index=False)
with (OUT/'stage2_protocol.json').open('w') as f:
    json.dump({'new_samples':48,'new_per_dataset':24,'new_time_limit_s':TIME_LIMIT_NEW,'old_time_limit_s':600,
               'old_instances':48,'retry_old_errors':len(retry_old),'thread_count':THREADS,'seed':SEED,
               'original_families':FAMILIES,'synthetic_prime_root_population':6068,
               'selection':'bounded finite LP/SDPprime results; stratified across size, relative root differences and families',
               'warning':'Old 600s and new 300s outcomes must be reported as separate time-limit cohorts or harmonized with solver logs; no claim of random representative prevalence from stratified samples.'},f,indent=2)
print('\nDONE — results in',OUT)
print(summary.to_string(index=False))
print('SEND stage2_all_results_with_original_and_retries.csv, stage2_manifest.csv, stage2_status_summary.csv')
if files_deferred:
    print('PARTIAL STAGE 2 ONLY — deferred files:',files_deferred)
    print('Prime pickle could not be loaded in this fresh kernel. Stop: do not infer unsolvability; an instance-level export or higher-memory machine is needed.')
else:
    print('All Stage 2 pickle files were loaded successfully.')


Prior benchmark: 48 rows | errors to retry: 3 | original 600s results protected
Stage 2 manifest: C:\Users\suuser\Desktop\minlplib\paperexperiments\reviewer_synthetic_difficulty\stage2_global_extension\stage2_manifest.csv new: 48 retries: 3
Loading instances_mixed_8016_candidate.pkl for 25 unfinished jobs


SystemError: deallocated bytearray object has exported buffers

MEMORY LIMIT: cannot load instances_mixed_8016_candidate.pkl | all previously solved instances are checkpointed; file deferred.
Stage 2 new/retry jobs completed: 26/51.

DONE — results in C:\Users\suuser\Desktop\minlplib\paperexperiments\reviewer_synthetic_difficulty\stage2_global_extension
        dataset               source     status  count
      Synthetic             new_300s    optimal     24
      Synthetic        original_600s      error      2
      Synthetic        original_600s    optimal     22
      Synthetic retry_old_error_300s    optimal      2
Synthetic_prime        original_600s      error      1
Synthetic_prime        original_600s    optimal     18
Synthetic_prime        original_600s time_limit      5
SEND stage2_all_results_with_original_and_retries.csv, stage2_manifest.csv, stage2_status_summary.csv
PARTIAL STAGE 2 ONLY — deferred files: ['instances_mixed_8016_candidate.pkl']
Prime pickle could not be loaded in this fresh kernel. Stop: do not infer unsolvability;

In [7]:
# STAGE 4 — Publication tables: sampled global QCQP optimality and relaxation gaps
# ONE CELL. No Gurobi, no pickle loading, no new optimization.
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json

PAPER=Path(r'C:\Users\suuser\Desktop\minlplib\paperexperiments')
BASE=PAPER/'reviewer_synthetic_difficulty'
OUT=BASE/'stage4_manuscript_global_optimum'
OUT.mkdir(parents=True,exist_ok=True)
SRC=BASE/'stage2_global_extension'/'stage2_all_results_with_original_and_retries.csv'
ALT=BASE/'global_qcqp_results.csv'
if not SRC.exists():
    raise FileNotFoundError(f'Stage 2 combined results missing: {SRC}. Do NOT substitute only the initial 48-run file {ALT}.')
df=pd.read_csv(SRC,low_memory=False)
required={'dataset','pickle_file','instance_id','status','best_feasible_obj','lp_bound','sdp_bound','sdpp_bound','source','time_limit_s'}
assert required.issubset(df.columns),f'Missing {required-set(df.columns)}'
# Prefer successfully retried runs for repeated instance IDs; keep one record per original instance.
df['status']=df['status'].astype(str).str.lower().str.strip()
df['dataset']=df['dataset'].astype(str)
df['_rank']=df['status'].map({'optimal':4,'time_limit':3,'infeasible':2,'unbounded':2,'other':1,'error':0}).fillna(1)
df['_position']=np.arange(len(df))
df=df.sort_values(['_rank','_position'],ascending=[False,False]).drop_duplicates(['dataset','pickle_file','instance_id']).drop(columns=['_rank','_position']).copy()
for c in ['best_feasible_obj','best_global_bound','runtime_s','solver_runtime_s','nodes','gurobi_mip_gap','lp_bound','sdp_bound','sdpp_bound','time_limit_s']:
    if c in df: df[c]=pd.to_numeric(df[c],errors='coerce')
assert df.groupby('dataset').size().to_dict()=={'Synthetic':48,'Synthetic_prime':24},'Unexpected sample sizes: check Stage 2 checkpoint and instance keys.'
assert (df.status.eq('optimal').sum()==66), 'Unexpected optimal count; audit records before publication.'
EPS=1e-3
LP_GAP_TOL=1e-5
VALIDITY_TOL=1e-4
# IMPORTANT: objective and Gurobi global bound should agree when optimal; solcount may be missing in old logs.
cert=df.status.eq('optimal') & np.isfinite(df.best_feasible_obj)
df['eligible_global_optimum']=cert
# Bound consistency checks: substantial differences between Gurobi primal and dual certificates should be audited.
df['gurobi_primal_dual_absolute_discrepancy']=np.where(cert & np.isfinite(df.best_global_bound),
                    np.abs(df.best_feasible_obj-df.best_global_bound),np.nan)
if (df['gurobi_primal_dual_absolute_discrepancy']>1e-3).any():
    print('AUDIT WARNING: optimal runs with primal/dual discrepancy >1e-3:',
          df.loc[df['gurobi_primal_dual_absolute_discrepancy']>1e-3,['dataset','instance_id','gurobi_primal_dual_absolute_discrepancy']].to_string(index=False))
for name,col in [('LP','lp_bound'),('SDP','sdp_bound'),('SDPprime','sdpp_bound')]:
    finite=cert & np.isfinite(df[col])
    df[f'{name}_valid_for_gap']=finite
    df[f'{name}_bound_violation']=np.where(finite,df[col]-df.best_feasible_obj,np.nan)
    df[f'{name}_normalized_opt_gap']=np.where(finite,np.maximum(0,df.best_feasible_obj-df[col])/(np.abs(df.best_feasible_obj)+EPS),np.nan)
    df[f'{name}_exact_at_1e5']=np.where(finite,(df.best_feasible_obj-df[col])<=LP_GAP_TOL,np.nan)
    df[f'{name}_paper_normalized_gap']=np.where(finite,np.maximum(0,df.best_feasible_obj-df[col])/(np.abs(df.best_feasible_obj)+np.abs(df[col])+EPS),np.nan)
    if (df[f'{name}_bound_violation']>VALIDITY_TOL).any():
        print(f'WARNING: {name} exceeds globally optimal objective by >{VALIDITY_TOL} for:',
              df.loc[df[f'{name}_bound_violation']>VALIDITY_TOL,['dataset','pickle_file','instance_id',col,'best_feasible_obj']].to_string(index=False))
for name,col in [('SDP','sdp_bound'),('SDPprime','sdpp_bound')]:
    den=df.best_feasible_obj-df.lp_bound
    mask=cert & np.isfinite(df.lp_bound) & np.isfinite(df[col]) & (den>LP_GAP_TOL)
    df[f'{name}_fraction_LP_gap_closed']=np.where(mask,(df[col]-df.lp_bound)/den,np.nan)
    df[f'{name}_relative_to_LP_sign']=np.where(mask,np.sign(df[col]-df.lp_bound),np.nan)

rows=[]
for ds,g in df.groupby('dataset',sort=True):
    opt=g[g.eligible_global_optimum]
    row={'dataset':ds,'sampled_distinct_instances':len(g),'proven_optimal':len(opt),
         'time_limited':int(g.status.eq('time_limit').sum()),'unresolved_errors':int(g.status.eq('error').sum()),
         'time_limit_cohorts':'; '.join(f'{int(t)}s:{n}' for t,n in g.groupby('time_limit_s').size().items()),
         'median_runtime_optimal_s':opt.runtime_s.median(),'max_runtime_optimal_s':opt.runtime_s.max(),
         'median_nodes_optimal':opt.nodes.median(),
         'mean_runtime_all_including_censored_s':g.runtime_s.mean()}
    for R in ['LP','SDP','SDPprime']:
        r=opt[f'{R}_normalized_opt_gap'].dropna()
        row[f'{R}_valid_optimal_pairs']=len(r)
        row[f'{R}_median_opt_gap']=r.median()
        row[f'{R}_q90_opt_gap']=r.quantile(.90) if len(r) else np.nan
        row[f'{R}_exact_count']=int(opt[f'{R}_exact_at_1e5'].fillna(0).sum())
        row[f'{R}_violation_count_gt_1e4']=int((opt[f'{R}_bound_violation']>VALIDITY_TOL).sum())
    for R in ['SDP','SDPprime']:
        q=opt[f'{R}_fraction_LP_gap_closed'].dropna()
        row[f'{R}_gap_closure_n']=len(q)
        row[f'{R}_median_fraction_LP_gap_closed']=q.median()
        row[f'{R}_improves_LP_n']=int((q>1e-8).sum())
        row[f'{R}_weakens_LP_n']=int((q< -1e-8).sum())
    rows.append(row)
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'stage4_global_optimum_summary.csv',index=False)
df.to_csv(OUT/'stage4_global_optimum_per_instance.csv',index=False)
# Time limit results are CENSORED, not proof of solving failure; old/new time limits differ.
fail=df[df.status.ne('optimal')][['dataset','pickle_file','instance_id','id_global','status','time_limit_s','runtime_s','nodes','best_feasible_obj','best_global_bound','gurobi_mip_gap','error']]
fail.to_csv(OUT/'stage4_timeouts_and_errors.csv',index=False)
# Compare gaps on same subset for paired LP vs SDP/SDPprime; full SDP aggregate coverage differs.
paired=[]
for ds,g in df[df.eligible_global_optimum].groupby('dataset'):
    for R in ['SDP','SDPprime']:
        m=g[f'{R}_valid_for_gap'] & g['LP_valid_for_gap']
        x=g.loc[m]
        paired.append({'dataset':ds,'comparison':f'LP versus {R}','paired_n':len(x),
                       'LP_median_gap_same_subset':x.LP_normalized_opt_gap.median(),
                       'alternative_median_gap_same_subset':x[f'{R}_normalized_opt_gap'].median()})
pd.DataFrame(paired).to_csv(OUT/'stage4_paired_gap_summary.csv',index=False)
# One figure; compare only LP and SDPprime within each dataset (full SDP missing for many cases).
fig,ax=plt.subplots(figsize=(9,5))
labels=[];series=[]
for ds in ['Synthetic','Synthetic_prime']:
    g=df[(df.dataset==ds)&df.eligible_global_optimum]
    for R in ['LP','SDPprime']:
        labels.append(f'{ds.replace("Synthetic_prime","Synthetic′")}\n{R}')
        series.append(g[f'{R}_normalized_opt_gap'].dropna().to_numpy())
ax.boxplot(series,tick_labels=labels,showfliers=False)
ax.set(ylabel='Gap relative to certified global optimum',title='Global-optimality gaps (globally solved QCQPs only)')
fig.tight_layout();fig.savefig(OUT/'stage4_global_optimality_gaps.png',dpi=200);plt.close(fig)
audit={'source_csv':str(SRC),'global_objective_definition':'Gurobi feasible objective only when status=optimal',
       'normalized_opt_gap':'max(0,zstar-L)/(|zstar|+0.001)',
       'fraction_LP_gap_closed':'(L_alt-L_LP)/(zstar-L_LP); only zstar-L_LP>1e-5',
       'paper_gap_alternative':'max(0,zstar-L)/(|zstar|+|L|+0.001)',
       'optimal_status_not_time_limit':True,
       'different_time_limits_must_be_disclosed':True,
       'subset_bias':'Stratified bounded finite LP/SDPprime sample; not all generated instances; unequal samples',
       'full_SDP_missing':'Use paired subset comparisons; -inf not counted as finite',
       'no_new_optimization':True}
(OUT/'stage4_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
print('\nSTAGE 4 RESULTS\n',summary.to_string(index=False))
print('\nPAIRED COMPARISONS\n',pd.DataFrame(paired).to_string(index=False))
print('\nTIMEOUTS/ERRORS\n',fail.to_string(index=False))
print('\nSaved:',OUT)


AUDIT WARNING: optimal runs with primal/dual discrepancy >1e-3:         dataset  instance_id  gurobi_primal_dual_absolute_discrepancy
      Synthetic        24529                                 0.001124
      Synthetic         4527                                 0.002771
Synthetic_prime         5937                                 0.110851
Synthetic_prime         5540                                 0.002274
Synthetic_prime         4590                                 0.004536
Synthetic_prime         4157                                 0.013381
Synthetic_prime         3446                                 0.027225
Synthetic_prime         2023                                 0.013450
Synthetic_prime         1368                                 0.001420
Synthetic_prime          685                                 0.015419
      Synthetic         9825                                 0.001179
      Synthetic         9416                                 0.003832
      Synthetic         55

In [8]:
# STAGE 5 — Manuscript-consistent root-node delta, family coverage, B&B separation
# ONE CELL. CSV-only; no pickle, no Gurobi. Uses STAGE 1 outputs created on your Windows machine.
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, json
PAPER=Path(r'C:\Users\suuser\Desktop\minlplib\paperexperiments')
S1=PAPER/'reviewer_synthetic_difficulty'/'stage1_root_distributions'
OUT=PAPER/'reviewer_synthetic_difficulty'/'stage5_manuscript_root_delta'
OUT.mkdir(parents=True,exist_ok=True)
ITEMS=S1/'root_bound_instance_level.csv'
SUMM=S1/'root_bound_summary.csv'
AUDIT=S1/'analysis_audit.json'
for p in [ITEMS,SUMM,AUDIT]:
    if not p.exists():raise FileNotFoundError(f'Run the revised Stage 1 first. Missing: {p}')
meta=json.loads(AUDIT.read_text(encoding='utf-8'))
files=meta.get('original_family_files',[])
assert len(files)==11,f'Expected 11 non-a files, found {len(files)}: {files}'
assert all(not Path(x).stem.removeprefix('instancesobj_').endswith('a') for x in files)
assert all('1200' not in x for x in files)
assert all('mixed_train8' not in x for x in files)
# Stage 1 output stores only valid finite LP/alternative root pairs, exactly as required.
items=pd.read_csv(ITEMS,low_memory=False)
need={'dataset','key','comparison','lp_obj','sdp_obj','sdp_prime_obj'}
assert need.issubset(items.columns),f'Missing {need-set(items.columns)}'
refsum=pd.read_csv(SUMM,low_memory=False)
for c in ['lp_obj','sdp_obj','sdp_prime_obj']:
    items[c]=pd.to_numeric(items[c],errors='coerce')
assert not items[['dataset','key','comparison']].duplicated().any(), 'Duplicate instance/comparison records.'
# IMPORTANT CORRECTION: Stage 1 descriptive measure used max(|L1|,|L2|,eta).
# The PAPER target instead is delta=(L_LP-L_alt)/(|L_LP|+|L_alt|+eta).
ETA=1e-3
TOLS=[0.001,0.01,0.05]
parts=[]
for comparison,altcol in [('LP_vs_SDP','sdp_obj'),('LP_vs_SDPprime','sdp_prime_obj')]:
    a=items[items.comparison.eq(comparison)].copy()
    good=np.isfinite(a.lp_obj)&np.isfinite(a[altcol])
    if not good.all(): raise ValueError(f'Unexpected nonfinite valid-pair records: {comparison}')
    a['paper_delta']=(a.lp_obj-a[altcol])/(np.abs(a.lp_obj)+np.abs(a[altcol])+ETA)
    # With delta negative, alternative relaxation gives the stronger lower bound.
    a['paper_label_LP_at_epsilon_0p01']=(a.paper_delta > -0.01).astype(int)
    a['comparison_alternative']=comparison.removeprefix('LP_vs_')
    parts.append(a)
res=pd.concat(parts,ignore_index=True)
# Generate per-family records from original aggregate details; Stage 1 wrote only pooled instance details.
original_details=res[res.dataset.eq('Synthetic_original_all_families')].copy()
family_records=[]
for fam, group in original_details.groupby('family',dropna=True):
    piece=group.copy()
    piece['dataset']='Synthetic_original_family_'+str(fam)
    family_records.append(piece)
if family_records:
    res=pd.concat([res]+family_records,ignore_index=True)
# Statistical reporting always conditions on finite paired bounds; total varies for full SDP.
combined=[]
for (dataset,comp),g in res.groupby(['dataset','comparison'],sort=True):
    src=refsum[(refsum.dataset==dataset)&(refsum.comparison==comp)]
    if len(src)!=1:raise RuntimeError(f'Missing Stage 1 summary row {dataset} / {comp}')
    total=int(src.iloc[0]['total_rows']); valid=int(src.iloc[0]['valid_finite_pairs'])
    if len(g)!=valid:raise RuntimeError(f'Mismatched paired finite N {dataset}/{comp}: {len(g)} vs {valid}')
    x=g.paper_delta.to_numpy()
    row={'dataset':dataset,'comparison':comp,'total_rows':total,'paired_finite_n':valid,
         'nonfinite_or_unsuccessful_pair_n':total-valid,'paired_coverage_pct':100*valid/total,
         'median_paper_delta':np.median(x),'median_abs_paper_delta':np.median(np.abs(x)),
         'q90_abs_paper_delta':np.quantile(np.abs(x),.90),
         'LP_label_fraction_epsilon_0p01':np.mean(x > -0.01),
         'alternative_label_fraction_epsilon_0p01':np.mean(x <= -0.01),
         'LP_strictly_stronger_fraction':np.mean(x>0),'alternative_strictly_stronger_fraction':np.mean(x<0)}
    for t in TOLS:
        row[f'near_tie_abs_delta_le_{t:g}']=np.mean(np.abs(x)<=t)
        row[f'LP_stronger_outside_{t:g}']=np.mean(x>t)
        row[f'alternative_stronger_outside_{t:g}']=np.mean(x< -t)
    combined.append(row)
summary=pd.DataFrame(combined)
summary.to_csv(OUT/'stage5_root_delta_summary_and_families.csv',index=False)
res[['dataset','family','key','comparison','lp_obj','sdp_obj','sdp_prime_obj','paper_delta','paper_label_LP_at_epsilon_0p01']].to_csv(OUT/'stage5_root_delta_instance_level.csv',index=False)
main=summary[summary.dataset.isin(['Synthetic_original_all_families','Synthetic_prime_final_root','Combined_final_root'])].copy()
main.to_csv(OUT/'stage5_root_delta_main_table.csv',index=False)
fam=summary[summary.dataset.str.startswith('Synthetic_original_family_')].copy()
fam.to_csv(OUT/'stage5_root_delta_family_table.csv',index=False)
for comp in ['LP_vs_SDP','LP_vs_SDPprime']:
    fig,ax=plt.subplots(figsize=(9,5))
    for ds,label in [('Synthetic_original_all_families','Original Synthetic (11 CSV families)'),('Synthetic_prime_final_root','Synthetic′ (6068)')]:
        x=res.loc[(res.dataset==ds)&(res.comparison==comp),'paper_delta'].to_numpy()
        if len(x):ax.hist(x,bins=np.linspace(-1,1,81),alpha=.35,density=True,label=f'{label}, N={len(x):,}')
    for v in [-.01,0,.01]:ax.axvline(v,c='black',lw=.9,ls='--')
    ax.set(xlim=(-1,1),xlabel=r'Manuscript delta = (L_LP - L_alt)/(|L_LP|+|L_alt|+0.001)',ylabel='Density',title=f'{comp}: root-node target distribution')
    ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/f'stage5_paper_delta_{comp}.png',dpi=180);plt.close(fig)
# Do not mislabel tree-level delta values as root targets.
bb=S1/'BB_target_delta_SEPARATE_from_root.csv'
if bb.exists():
    b=pd.read_csv(bb)
    b.to_csv(OUT/'stage5_BB_target_delta_SEPARATE.csv',index=False)
else:print('NOTE: Stage 1 BB_target_delta_SEPARATE_from_root.csv unavailable; no B&B distribution generated.')
# Don't fabricate Combined root values; diagnostic distinguishes absent root source.
if not (main.dataset=='Combined_final_root').any():
    print('COMBINED ROOT NOT AVAILABLE: Stage 1 found no schema-verified combined root objective file.')
(OUT/'stage5_audit.json').write_text(json.dumps({
    'source':str(ITEMS),'stage1_audit':str(AUDIT),'finite_pair_reporting':True,
    'paper_delta_formula':'(L_LP-L_alternative)/(abs(L_LP)+abs(L_alternative)+0.001)',
    'paper_class_label':'1 if delta > -0.01 else 0',
    'descriptive_stage1_formula_replaced':'(L_alt-L_LP)/max(abs(L_alt),abs(L_LP),0.001)',
    'original_families_included':files,'synthetic_prime_root_population':6068,
    'combined_root_available':bool((main.dataset=='Combined_final_root').any()),
    'BB_delta_never_used_as_root_delta':True,
    'not_all_manuscript_training_data_verified':'Original 11-family CSV aggregate may differ from the manuscript specific training/test membership',
    'no_new_optimization':True},indent=2),encoding='utf-8')
print('\nMAIN ROOT-NODE DELTA SUMMARY (MANUSCRIPT FORMULA)\n',main.to_string(index=False))
print('\nORIGINAL FAMILY COVERAGE\n',fam[['dataset','comparison','total_rows','paired_finite_n','median_abs_paper_delta','near_tie_abs_delta_le_0.01']].to_string(index=False))
print('\nSaved:',OUT)


COMBINED ROOT NOT AVAILABLE: Stage 1 found no schema-verified combined root objective file.

MAIN ROOT-NODE DELTA SUMMARY (MANUSCRIPT FORMULA)
                         dataset     comparison  total_rows  paired_finite_n  nonfinite_or_unsuccessful_pair_n  paired_coverage_pct  median_paper_delta  median_abs_paper_delta  q90_abs_paper_delta  LP_label_fraction_epsilon_0p01  alternative_label_fraction_epsilon_0p01  LP_strictly_stronger_fraction  alternative_strictly_stronger_fraction  near_tie_abs_delta_le_0.001  LP_stronger_outside_0.001  alternative_stronger_outside_0.001  near_tie_abs_delta_le_0.01  LP_stronger_outside_0.01  alternative_stronger_outside_0.01  near_tie_abs_delta_le_0.05  LP_stronger_outside_0.05  alternative_stronger_outside_0.05
Synthetic_original_all_families      LP_vs_SDP      320500           166710                            153790            52.015601           -0.277088                0.386414             0.927072                        0.236345                   